# Model inference

In [7]:
# 1. install and imports
!pip install -q rasterio torchmetrics scipy pandas

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import rasterio
import pandas as pd
from pathlib import Path
from dataclasses import dataclass, field
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings("ignore")

# 2. Configuration

In [8]:
@dataclass
class Config:
    # data path
    ORIGINAL_DATA_ROOT: Path = Path("/kaggle/input/quezon-city-informal-settlements/qc")
    
    # system
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    NUM_WORKERS: int = 2
    BATCH_SIZE: int = 16 
    
    # data split (seed 42 - same as training to ensure consistent test set)
    VAL_SPLIT: float = 0.25
    TEST_SPLIT: float = 0.25
    RANDOM_SEED: int = 42
    
    # architecture defaults
    ENCODER_CHANNEL_LIST: list = field(default_factory=lambda: [80, 160, 320, 640])
    ENCODER_BLOCKS_PER_STAGE: list = field(default_factory=lambda: [2, 2, 8, 2])
    DECODER_CONVNEXT_BLOCKS: list = field(default_factory=lambda: [2, 2, 2, 2])
    
    # UPDATED: Last element changed from 20 to 64 to match UNetDecoder weights
    FINAL_UPSAMPLING_CHANNELS: list = field(default_factory=lambda: [80, 40, 64])
    
    UNET_DECODER_CHANNEL_LIST: list = field(default_factory=lambda: [512, 256, 128, 64]) 
    
    # stats
    RGB_MEAN: list = field(default_factory=lambda: [0.33969313, 0.35239491, 0.28135468])
    RGB_STD: list = field(default_factory=lambda: [0.23594516, 0.20353660, 0.20314776])
    BC_MEAN: list = field(default_factory=lambda: [0.0009436231339350343])
    BC_STD: list = field(default_factory=lambda: [0.001719754422083497])
    BH_MEAN: list = field(default_factory=lambda: [3.086625337600708])
    BH_STD: list = field(default_factory=lambda: [5.610204696655273])
    
    # dynamic fields
    MODALITY_TO_RUN: str = "all" 
    INPUT_CHANNELS: int = 5
    ENCODER_DROP_PATH_RATE: float = 0.0
    ENCODER_LAYER_SCALE_INIT_VALUE: float = 1e-6

config = Config()

torch.manual_seed(config.RANDOM_SEED)
np.random.seed(config.RANDOM_SEED)
random.seed(config.RANDOM_SEED)

# 3. Model Architecture

In [9]:
# --- components ---
class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-6, data_format="channels_last"):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))
        self.eps = eps
        self.data_format = data_format
        self.normalized_shape = (normalized_shape,)
    def forward(self, x):
        if self.data_format == "channels_last":
            return F.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        elif self.data_format == "channels_first":
            u = x.mean(1, keepdim=True)
            s = (x - u).pow(2).mean(1, keepdim=True)
            x = (x - u) / torch.sqrt(s + self.eps)
            x = self.weight[:, None, None] * x + self.bias[:, None, None]
            return x

class ConvNeXtBlock(nn.Module):
    def __init__(self, dim, drop_path_rate=0., layer_scale_init_value=1e-6):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = LayerNorm(dim, eps=1e-6, data_format="channels_first")
        self.pwconv1 = nn.Conv2d(dim, 4 * dim, kernel_size=1)
        self.act = nn.GELU()
        self.pwconv2 = nn.Conv2d(4 * dim, dim, kernel_size=1)
        self.gamma = nn.Parameter(layer_scale_init_value * torch.ones((dim, 1, 1)), requires_grad=True) if layer_scale_init_value > 0 else None
    def forward(self, x):
        input = x
        x = self.dwconv(x)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        if self.gamma is not None: x = self.gamma * x
        return input + x

# --- encoders ---
class ConvNeXtEncoder(nn.Module):
    def __init__(self, config: Config, in_chans: int):
        super().__init__()
        self.dims = config.ENCODER_CHANNEL_LIST
        self.depths = config.ENCODER_BLOCKS_PER_STAGE
        self.stem = nn.Sequential(nn.Conv2d(in_chans, self.dims[0], 4, 4), LayerNorm(self.dims[0], eps=1e-6, data_format="channels_first"))
        self.stages = nn.ModuleList()
        self.downsamplers = nn.ModuleList()
        for i in range(4):
            if i > 0:
                self.downsamplers.append(nn.Sequential(LayerNorm(self.dims[i-1], eps=1e-6, data_format="channels_first"), nn.Conv2d(self.dims[i-1], self.dims[i], 2, 2)))
            self.stages.append(nn.Sequential(*[ConvNeXtBlock(self.dims[i]) for _ in range(self.depths[i])]))
        self.output_channels = self.dims
    def forward(self, x):
        features = {}
        x = self.stem(x)
        x = self.stages[0](x); features['s1'] = x
        x = self.downsamplers[0](x); x = self.stages[1](x); features['s2'] = x
        x = self.downsamplers[1](x); x = self.stages[2](x); features['s3'] = x
        x = self.downsamplers[2](x); x = self.stages[3](x); features['s4'] = x
        return features

class ConvNeXtEncoder_3Stage(nn.Module):
    def __init__(self, config: Config, in_chans: int):
        super().__init__()
        dims = config.ENCODER_CHANNEL_LIST[:3]
        depths = config.ENCODER_BLOCKS_PER_STAGE[:3]
        self.stem = nn.Sequential(nn.Conv2d(in_chans, dims[0], 4, 4), LayerNorm(dims[0], eps=1e-6, data_format="channels_first"))
        self.stage0 = nn.Sequential(*[ConvNeXtBlock(dims[0]) for _ in range(depths[0])])
        self.downsampler1 = nn.Sequential(LayerNorm(dims[0], eps=1e-6, data_format="channels_first"), nn.Conv2d(dims[0], dims[1], 2, 2))
        self.stage1 = nn.Sequential(*[ConvNeXtBlock(dims[1]) for _ in range(depths[1])])
        self.downsampler2 = nn.Sequential(LayerNorm(dims[1], eps=1e-6, data_format="channels_first"), nn.Conv2d(dims[1], dims[2], 2, 2))
        self.stage2 = nn.Sequential(*[ConvNeXtBlock(dims[2]) for _ in range(depths[2])])
    def forward(self, x):
        s1 = self.stage0(self.stem(x))
        s2 = self.stage1(self.downsampler1(s1))
        s3 = self.stage2(self.downsampler2(s2))
        return {'s1': s1, 's2': s2, 's3': s3}

# --- decoders ---
class ConvNeXtDecoder(nn.Module):
    def __init__(self, config: Config, encoder_channels: list[int]):
        super().__init__()
        s1_ch, s2_ch, s3_ch, s4_ch = encoder_channels
        self.bottleneck = nn.Sequential(*[ConvNeXtBlock(s4_ch) for _ in range(config.DECODER_CONVNEXT_BLOCKS[0])])
        self.up1 = nn.ConvTranspose2d(s4_ch, s3_ch, 2, 2)
        self.dec_block1 = nn.Sequential(nn.Conv2d(s3_ch*2, s3_ch, 1), *[ConvNeXtBlock(s3_ch) for _ in range(config.DECODER_CONVNEXT_BLOCKS[1])])
        self.up2 = nn.ConvTranspose2d(s3_ch, s2_ch, 2, 2)
        self.dec_block2 = nn.Sequential(nn.Conv2d(s2_ch*2, s2_ch, 1), *[ConvNeXtBlock(s2_ch) for _ in range(config.DECODER_CONVNEXT_BLOCKS[2])])
        self.up3 = nn.ConvTranspose2d(s2_ch, s1_ch, 2, 2)
        self.dec_block3 = nn.Sequential(nn.Conv2d(s1_ch*2, s1_ch, 1), *[ConvNeXtBlock(s1_ch) for _ in range(config.DECODER_CONVNEXT_BLOCKS[3])])
        self.final_up1 = nn.ConvTranspose2d(s1_ch, 40, 2, 2)
        self.final_conv1 = nn.Sequential(nn.Conv2d(40, 40, 3, 1, 1), LayerNorm(40, data_format="channels_first"), nn.GELU())
        self.final_up2 = nn.ConvTranspose2d(40, 20, 2, 2)
        self.final_conv_out = nn.Conv2d(20, 1, 1)
    def forward(self, features: dict):
        x = self.bottleneck(features['s4'])
        x = self.up1(x); x = torch.cat([x, features['s3']], dim=1); x = self.dec_block1(x)
        x = self.up2(x); x = torch.cat([x, features['s2']], dim=1); x = self.dec_block2(x)
        x = self.up3(x); x = torch.cat([x, features['s1']], dim=1); x = self.dec_block3(x)
        x = self.final_up1(x); x = self.final_conv1(x); x = self.final_up2(x)
        return self.final_conv_out(x)

class UNetDecoder(nn.Module):
    """Classic U-Net style decoder fed by ConvNeXt encoder skips."""
    def __init__(self, config: Config, encoder_channels: list[int]):
        super().__init__()
        s1_ch, s2_ch, s3_ch, s4_ch = encoder_channels
        d4_ch, d3_ch, d2_ch, d1_ch = config.UNET_DECODER_CHANNEL_LIST

        def double_conv(in_ch: int, out_ch: int) -> nn.Sequential:
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
            )

        self.bottleneck = double_conv(s4_ch, s4_ch)
        self.up1 = nn.ConvTranspose2d(s4_ch, d4_ch, kernel_size=2, stride=2)
        self.dec_block1 = double_conv(d4_ch + s3_ch, d4_ch)
        self.up2 = nn.ConvTranspose2d(d4_ch, d3_ch, kernel_size=2, stride=2)
        self.dec_block2 = double_conv(d3_ch + s2_ch, d3_ch)
        self.up3 = nn.ConvTranspose2d(d3_ch, d2_ch, kernel_size=2, stride=2)
        self.dec_block3 = double_conv(d2_ch + s1_ch, d2_ch)
        self.up4 = nn.ConvTranspose2d(d2_ch, d1_ch, kernel_size=2, stride=2)
        self.dec_block4 = double_conv(d1_ch, d1_ch)

        final_channels = config.FINAL_UPSAMPLING_CHANNELS[-1]
        self.final_up = nn.ConvTranspose2d(d1_ch, final_channels, kernel_size=2, stride=2)
        self.final_conv_out = nn.Conv2d(final_channels, 1, kernel_size=1)

    def forward(self, features: dict):
        s1, s2, s3, s4 = features["s1"], features["s2"], features["s3"], features["s4"]
        bottleneck = self.bottleneck(s4)
        x = self.up1(bottleneck); x = torch.cat([x, s3], dim=1); x = self.dec_block1(x)
        x = self.up2(x); x = torch.cat([x, s2], dim=1); x = self.dec_block2(x)
        x = self.up3(x); x = torch.cat([x, s1], dim=1); x = self.dec_block3(x)
        x = self.up4(x); x = self.dec_block4(x)
        x = self.final_up(x)
        return self.final_conv_out(x)

# --- fusion components ---
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False), 
            nn.ReLU(), 
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return x * self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x))).expand_as(x)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return x * self.sigmoid(self.conv1(torch.cat([torch.mean(x, 1, True), torch.max(x, 1, True)[0]], 1))).expand_as(x)

class CBAM(nn.Module):
    def __init__(self, in_planes):
        super().__init__()
        self.ca = ChannelAttention(in_planes)
        self.sa = SpatialAttention()
    def forward(self, x):
        return self.sa(self.ca(x))

class FusionBlock(nn.Module):
    def __init__(self, in_channels_list, out_channels):
        super().__init__()
        total_in = sum(in_channels_list)
        self.cbam = CBAM(total_in)
        self.compress_conv = nn.Sequential(nn.Conv2d(total_in, out_channels, 1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(True))
    def forward(self, features):
        return self.compress_conv(self.cbam(torch.cat(features, 1)))

# --- Models ---
class ConvNeXtUNet(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        self.encoder = ConvNeXtEncoder(config, in_chans=config.INPUT_CHANNELS)
        self.decoder = ConvNeXtDecoder(config, encoder_channels=self.encoder.output_channels)
    def forward(self, x): return self.decoder(self.encoder(x))

class ConvNeXtUNet_PlainDecoder(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        self.encoder = ConvNeXtEncoder(config, in_chans=config.INPUT_CHANNELS)
        self.decoder = UNetDecoder(config, encoder_channels=self.encoder.output_channels)
    def forward(self, x): return self.decoder(self.encoder(x))

class SettleNet(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        dims = config.ENCODER_CHANNEL_LIST
        self.encoder_rgb = ConvNeXtEncoder_3Stage(config, in_chans=3)
        self.encoder_bc = ConvNeXtEncoder_3Stage(config, in_chans=1)
        self.encoder_bh = ConvNeXtEncoder_3Stage(config, in_chans=1)
        self.fusion_blocks = nn.ModuleList([FusionBlock([d,d,d], d) for d in dims[:3]])
        self.bottleneck_bridge = nn.Sequential(LayerNorm(dims[2], eps=1e-6, data_format="channels_first"), nn.Conv2d(dims[2], dims[3], 2, 2))
        self.decoder = ConvNeXtDecoder(config, encoder_channels=dims)
    def forward(self, x):
        x_rgb, x_bc, x_bh = x[:, :3], x[:, 3:4], x[:, 4:5]
        f_rgb, f_bc, f_bh = self.encoder_rgb(x_rgb), self.encoder_bc(x_bc), self.encoder_bh(x_bh)
        s1 = self.fusion_blocks[0]([f_rgb['s1'], f_bc['s1'], f_bh['s1']])
        s2 = self.fusion_blocks[1]([f_rgb['s2'], f_bc['s2'], f_bh['s2']])
        s3 = self.fusion_blocks[2]([f_rgb['s3'], f_bc['s3'], f_bh['s3']])
        s4 = self.bottleneck_bridge(s3)
        return self.decoder({'s1':s1, 's2':s2, 's3':s3, 's4':s4})

# 4. Data Pipeline

In [10]:
class TestDataset(Dataset):
    def __init__(self, config: Config, image_ids: list, modality: str):
        self.root = config.ORIGINAL_DATA_ROOT
        self.ids = image_ids
        self.modality = modality
        self.dirs = {
            'sat': self.root / "satellite-256", 'mask': self.root / "mask-256",
            'bc': self.root / "bc-256", 'bh': self.root / "bh-256"
        }
        self.norm = A.Compose([
            A.Normalize(
                mean=config.RGB_MEAN + config.BC_MEAN + config.BH_MEAN,
                std=config.RGB_STD + config.BC_STD + config.BH_STD,
                max_pixel_value=1.0
            ),
            ToTensorV2()
        ])

    def __len__(self): return len(self.ids)

    def _load_img(self, k, img_id):
        path_list = list(self.dirs[k].glob(f"{img_id}.*"))
        if not path_list: raise FileNotFoundError(f"file {img_id} not found")
        path = path_list[0]
        if k == 'sat': return np.array(Image.open(path).convert("RGB"), dtype=np.float32) / 255.0
        with rasterio.open(path) as src: return np.expand_dims(src.read(1).astype(np.float32), axis=-1)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        sat = self._load_img('sat', img_id); bc = self._load_img('bc', img_id); bh = self._load_img('bh', img_id)
        full_img = np.concatenate([sat, bc, bh], axis=-1)
        mask_path = list(self.dirs['mask'].glob(f"{img_id}.*"))[0]
        mask = (np.array(Image.open(mask_path).convert("L")) > 0).astype(np.float32)
        aug = self.norm(image=full_img, mask=mask)
        full_tensor = aug['image']; mask_tensor = aug['mask'].unsqueeze(0)
        
        if self.modality == 'satellite': return full_tensor[:3, :, :], mask_tensor, img_id
        if self.modality == 'bc': return full_tensor[3:4, :, :], mask_tensor, img_id
        if self.modality == 'bh': return full_tensor[4:5, :, :], mask_tensor, img_id
        return full_tensor, mask_tensor, img_id

def get_test_loader(config, modality):
    all_files = sorted([f.stem for f in (config.ORIGINAL_DATA_ROOT / "satellite-256").iterdir()])
    total_size = len(all_files)
    test_size = int(config.TEST_SPLIT * total_size)
    g = torch.Generator().manual_seed(config.RANDOM_SEED)
    indices = torch.randperm(total_size, generator=g).tolist()
    test_ids = [all_files[i] for i in indices[-test_size:]]
    ds = TestDataset(config, test_ids, modality)
    return DataLoader(ds, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=config.NUM_WORKERS)

# 5. Global Metric Evaluation Engine

In [11]:
def run_global_evaluation(model_name, model_class, weights_path, modality, config):
    print(f"\n{'='*80}")
    print(f"EVALUATING MODEL: {model_name}")
    print(f"Modality: {modality} | Weights: {weights_path}")
    print(f"{'='*80}")
    
    # Configure input channels based on modality
    if modality == 'satellite': config.INPUT_CHANNELS = 3
    elif modality in ['bc', 'bh']: config.INPUT_CHANNELS = 1
    else: config.INPUT_CHANNELS = 5
        
    loader = get_test_loader(config, modality)
    
    model = model_class(config)
    try:
        model.load_state_dict(torch.load(weights_path, map_location=config.DEVICE))
    except Exception as e:
        print(f"Error loading weights: {e}")
        return None

    model.to(config.DEVICE)
    model.eval()
    
    # Initialize global counters (using float64 to prevent overflow)
    TP = 0.0
    TN = 0.0
    FP = 0.0
    FN = 0.0
    
    with torch.no_grad():
        for batch_images, batch_masks, _ in tqdm(loader, desc=f"Evaluating {model_name}"):
            batch_images = batch_images.to(config.DEVICE)
            batch_masks = batch_masks.to(config.DEVICE)
            
            # Run inference
            logits = model(batch_images)
            preds = torch.sigmoid(logits)
            
            # Binarize (Threshold 0.5)
            preds_bin = (preds > 0.5).float()
            targets_bin = (batch_masks > 0.5).float()
            
            # Accumulate Counts (Global)
            # TP: Predicted 1, Actual 1
            TP += (preds_bin * targets_bin).sum().item()
            # TN: Predicted 0, Actual 0
            TN += ((1 - preds_bin) * (1 - targets_bin)).sum().item()
            # FP: Predicted 1, Actual 0
            FP += (preds_bin * (1 - targets_bin)).sum().item()
            # FN: Predicted 0, Actual 1
            FN += ((1 - preds_bin) * targets_bin).sum().item()
            
    # Compute Metrics
    epsilon = 1e-7
    
    accuracy = (TP + TN) / (TP + TN + FP + FN + epsilon)
    precision = TP / (TP + FP + epsilon)
    recall = TP / (TP + FN + epsilon)
    f1_score = 2 * (precision * recall) / (precision + recall + epsilon)
    iou_target = TP / (TP + FP + FN + epsilon)
    iou_background = TN / (TN + FP + FN + epsilon)
    
    results = {
        "Model": model_name,
        "TP": int(TP),
        "TN": int(TN),
        "FP": int(FP),
        "FN": int(FN),
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1_score,
        "IoU (Target)": iou_target,
        "IoU (Background)": iou_background
    }
    
    print("-" * 40)
    print(f"Confusion Matrix (Pixels):")
    print(f"TP: {int(TP):<15} FP: {int(FP)}")
    print(f"FN: {int(FN):<15} TN: {int(TN)}")
    print("-" * 40)
    print(f"Global Metrics:")
    print(f"Accuracy:       {accuracy:.4f}")
    print(f"Precision:      {precision:.4f}")
    print(f"Recall:         {recall:.4f}")
    print(f"F1-Score:       {f1_score:.4f}")
    print(f"IoU (Target):   {iou_target:.4f}")
    print(f"IoU (Backgrnd): {iou_background:.4f}")
    print("-" * 40)
    
    del model
    torch.cuda.empty_cache()
    return results

# 6. Run Evaluation

In [12]:
# Define the 6 models to test
MODELS_TO_TEST = [
    {
        "name": "ConvNeXt Satellite",
        "class": ConvNeXtUNet,
        "path": '/kaggle/input/convnextunet-satellite-xqkdckas/pytorch/default/1/sat-xqkdckas.pth',
        "modality": "satellite"
    },
    {
        "name": "ConvNeXt Building Count",
        "class": ConvNeXtUNet,
        "path": '/kaggle/input/convnext-bc-g00wx4xl/pytorch/default/1/convnext-bc-g00wx4xl.pth',
        "modality": "bc"
    },
    {
        "name": "ConvNeXt Building Height",
        "class": ConvNeXtUNet,
        "path": '/kaggle/input/convnext-bh/pytorch/default/1/convnext-bh.pth',
        "modality": "bh"
    },
    {
        "name": "ConvNeXt Fusion All",
        "class": ConvNeXtUNet,
        "path": '/kaggle/input/convnext-all-7ir38hd4/pytorch/default/1/convnext-all-7ir38hd4.pth',
        "modality": "all"
    },
    {
        "name": "ConvNeXt UNet Plain Decoder Satellite",
        "class": ConvNeXtUNet_PlainDecoder,
        "path": '/kaggle/input/convnext-unet-plain-decoder-satellite/pytorch/default/1/unet-plain-decoder-satellite.pth',
        "modality": "satellite"
    },
    {
        "name": "SettleNet (CBAM)",
        "class": SettleNet,
        "path": '/kaggle/input/settlenet-rxrj9b9b/pytorch/default/1/settlenet-rxrj9b9b.pth',
        "modality": "all"
    }
]

# Run loop and collect data
all_results = []

for m in MODELS_TO_TEST:
    res = run_global_evaluation(
        model_name=m["name"],
        model_class=m["class"],
        weights_path=m["path"],
        modality=m["modality"],
        config=config
    )
    if res:
        all_results.append(res)

# Display final summary dataframe
print("\nFINAL SUMMARY TABLE:")
df_results = pd.DataFrame(all_results)
df_display = df_results[["Model", "Accuracy", "Precision", "Recall", "F1-Score", "IoU (Target)", "IoU (Background)"]]
print(df_display.to_string(index=False))

# Display Confusion Matrix data separately
print("\nCONFUSION MATRIX DATA:")
df_cm = df_results[["Model", "TP", "TN", "FP", "FN"]]
print(df_cm.to_string(index=False))


EVALUATING MODEL: ConvNeXt Satellite
Modality: satellite | Weights: /kaggle/input/convnextunet-satellite-xqkdckas/pytorch/default/1/sat-xqkdckas.pth


Evaluating ConvNeXt Satellite:   0%|          | 0/157 [00:00<?, ?it/s]

----------------------------------------
Confusion Matrix (Pixels):
TP: 4827836         FP: 5021077
FN: 1829853         TN: 152226770
----------------------------------------
Global Metrics:
Accuracy:       0.9582
Precision:      0.4902
Recall:         0.7252
F1-Score:       0.5850
IoU (Target):   0.4134
IoU (Backgrnd): 0.9569
----------------------------------------

EVALUATING MODEL: ConvNeXt Building Count
Modality: bc | Weights: /kaggle/input/convnext-bc-g00wx4xl/pytorch/default/1/convnext-bc-g00wx4xl.pth


Evaluating ConvNeXt Building Count:   0%|          | 0/157 [00:00<?, ?it/s]

----------------------------------------
Confusion Matrix (Pixels):
TP: 5021737         FP: 6313135
FN: 1635952         TN: 150934712
----------------------------------------
Global Metrics:
Accuracy:       0.9515
Precision:      0.4430
Recall:         0.7543
F1-Score:       0.5582
IoU (Target):   0.3872
IoU (Backgrnd): 0.9500
----------------------------------------

EVALUATING MODEL: ConvNeXt Building Height
Modality: bh | Weights: /kaggle/input/convnext-bh/pytorch/default/1/convnext-bh.pth


...

----------------------------------------
Confusion Matrix (Pixels):
TP: 4723974         FP: 3843260
FN: 1933715         TN: 153404587
----------------------------------------
Global Metrics:
Accuracy:       0.9648
Precision:      0.5514
Recall:         0.7096
F1-Score:       0.6206
IoU (Target):   0.4499
IoU (Backgrnd): 0.9637
----------------------------------------

EVALUATING MODEL: ConvNeXt UNet Plain Decoder Satellite
Modality: satellite | Weights: /kaggle/input/convnext-unet-plain-decoder-satellite/pytorch/default/1/unet-plain-decoder-satellite.pth


Evaluating ConvNeXt UNet Plain Decoder Satellite:   0%|          | 0/157 [00:00<?, ?it/s]

----------------------------------------
Confusion Matrix (Pixels):
TP: 4934606         FP: 5599721
FN: 1723083         TN: 151648126
----------------------------------------
Global Metrics:
Accuracy:       0.9553
Precision:      0.4684
Recall:         0.7412
F1-Score:       0.5741
IoU (Target):   0.4026
IoU (Backgrnd): 0.9539
----------------------------------------

EVALUATING MODEL: SettleNet (CBAM)
Modality: all | Weights: /kaggle/input/settlenet-rxrj9b9b/pytorch/default/1/settlenet-rxrj9b9b.pth


Evaluating SettleNet (CBAM):   0%|          | 0/157 [00:00<?, ?it/s]

----------------------------------------
Confusion Matrix (Pixels):
TP: 4530715         FP: 3305654
FN: 2126974         TN: 153942193
----------------------------------------
Global Metrics:
Accuracy:       0.9669
Precision:      0.5782
Recall:         0.6805
F1-Score:       0.6252
IoU (Target):   0.4547
IoU (Backgrnd): 0.9659
----------------------------------------

FINAL SUMMARY TABLE:
                                Model  Accuracy  Precision   Recall  F1-Score  IoU (Target)  IoU (Background)
                   ConvNeXt Satellite  0.958202   0.490190 0.725152  0.584958      0.413386          0.956933
              ConvNeXt Building Count  0.951502   0.443034 0.754276  0.558201      0.387156          0.949969
             ConvNeXt Building Height  0.951294   0.435289 0.669588  0.527596      0.358323          0.949933
                  ConvNeXt Fusion All  0.964754   0.551400 0.709552  0.620558      0.449862          0.963708
ConvNeXt UNet Plain Decoder Satellite  0.955323   0.468431